# 03. Poisson Attack & Defense Ratings

**Stage:** 03_ratings  
**Inputs:** `data/processed/clean_fixtures.parquet`, `src/config/config.yaml`  
**Outputs:** `data/processed/ratings.parquet`  

This notebook computes rolling point-in-time Poisson attack and defense ratings per team using GLM regression over past matches. Window length is read dynamically from configuration to prevent data leakage (.shift(1)).

In [1]:
# Load imports and configuration
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.config.loader import get_rating_window, load_config
from src.ratings.poisson import compute_rolling_ratings

config = load_config()
rating_window = get_rating_window(config)
PROJECT_ROOT = Path.cwd().resolve().parents[1]
processed_dir = PROJECT_ROOT / "data" / "processed"

clean_df = None
ratings_df = None
print(f"Ratings stage initialized with rolling window = {rating_window} matches")

Matplotlib is building the font cache; this may take a moment.


Ratings stage initialized with rolling window = 8 matches


In [2]:
# Load cleaned fixtures
clean_file = processed_dir / "clean_fixtures.parquet"
if clean_file.exists():
    clean_df = pd.read_parquet(clean_file)
    print(f"Loaded clean fixtures: {len(clean_df)} rows")
else:
    print(f"Warning: Clean fixtures file not found at {clean_file}")

Loaded clean fixtures: 380 rows


In [3]:
# Compute rolling Poisson ratings
if clean_df is not None and not clean_df.empty:
    ratings_df = compute_rolling_ratings(clean_df, window=rating_window)
    print(f"Computed ratings for {len(ratings_df)} fixtures")
else:
    print("Skipping rating calculation: clean fixtures not available")

/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/statsmodels/genmod/generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/statsmodels/ge

Computed ratings for 380 fixtures


/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/statsmodels/genmod/families/family.py:143: RuntimeWarning: divide by zero encountered in divide
  return 1. / (self.link.deriv(mu)**2 * self.variance(mu))
/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/statsmodels/regression/_tools.py:121: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/statsmodels/genmod/families/family.py:445: RuntimeWarning: invalid value encountered in divide
  endog_mu = self._clean(endog / mu)
/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/lib/python3.14/site-packages/statsmodels/genmod/families/family.py:143: RuntimeWarning: divide by zero encountered in divide
  return 1. / (self.link.deriv(mu)**2 * self.variance(mu))
/Users/mac/Documents/Projects/SportsBettingPoisson+ML/.venv/

In [4]:
# Summary display
if ratings_df is not None:
    print("=== Sample Rolling Ratings ===")
    display(ratings_df[["date", "home_team", "away_team", "home_attack_rating", "home_defense_rating", "expected_home_goals", "expected_away_goals"]].head())
    print("\n=== Expected Goals Summary ===")
    display(ratings_df[["expected_home_goals", "expected_away_goals"]].describe())

=== Sample Rolling Ratings ===


,date,home_team,away_team,home_attack_rating,home_defense_rating,expected_home_goals,expected_away_goals
0,2024-08-16,Manchester United,Fulham,1.0,1.0,1.509,1.207
1,2024-08-17,Arsenal,Wolverhampton Wanderers,1.0,1.0,1.509,1.207
2,2024-08-17,Everton,Brighton & Hove Albion,1.0,1.0,1.509,1.207
3,2024-08-17,Ipswich Town,Liverpool,1.0,1.0,1.509,1.207
4,2024-08-17,Newcastle United,Southampton,1.0,1.0,1.509,1.207



=== Expected Goals Summary ===


,expected_home_goals,expected_away_goals
count,3.800000e+02,3.800000e+02
mean,2.107787e+13,4.278091e+37
std,4.108834e+14,8.333334e+38
min,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00
50%,8.520000e-01,8.175000e-01
75%,1.515750e+00,1.437750e+00
max,8.009591e+15,1.624469e+40


In [5]:
# Save ratings dataset
if ratings_df is not None:
    output_file = processed_dir / "ratings.parquet"
    ratings_df.to_parquet(output_file, index=False)
    print(f"Saved Poisson ratings to {output_file}")

Saved Poisson ratings to /Users/mac/Documents/Projects/SportsBettingPoisson+ML/data/processed/ratings.parquet
